In [ ]:
# 1. 安裝必要套件
!pip install pandas numpy scikit-learn xgboost shap matplotlib openpyxl

Bidirectional Elimination（雙向逐步法）

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import shap
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import xgboost as xgb

# 1. 載入資料
import sys
from pathlib import Path
_root = Path.cwd().resolve()
for _ in range(10):
    if (_root / "src" / "data_layout.py").exists():
        break
    _root = _root.parent
sys.path.insert(0, str(_root / "src"))
from data_layout import resolve_processed_data_dir
_csv_dir = resolve_processed_data_dir(_root)
df = pd.read_csv(_csv_dir / "2025_metrics.csv")
if "Season" in df.columns:
    df = df[df["Season"] == 2025].reset_index(drop=True)

# 2. 準備特徵與目標
target = 'Win Rate'
exclude_cols = [c for c in ['Team', 'Season', target] if c in df.columns]
X = df.drop(columns=exclude_cols)
y = df[target]

# 3. 標準化資料
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

# 4. 雙向逐步法（Bidirectional Elimination）直接使用所有特徵
def stepwise_selection(X, y, initial_list=[], threshold_in=0.05, threshold_out=0.10, verbose=True):
    included = list(initial_list)
    while True:
        changed=False
        # forward step
        excluded = list(set(X.columns) - set(included))
        new_pval = pd.Series(index=excluded)
        for new_col in excluded:
            model = sm.OLS(y, sm.add_constant(pd.DataFrame(X[included + [new_col]]))).fit()
            new_pval[new_col] = model.pvalues[new_col]
        best_pval = new_pval.min()
        if best_pval < threshold_in:
            best_feature = new_pval.idxmin()
            included.append(best_feature)
            changed=True
            if verbose:
                print('Add  {:30} with p-value {:.6}'.format(best_feature, best_pval))
        # backward step
        model = sm.OLS(y, sm.add_constant(pd.DataFrame(X[included]))).fit()
        pvalues = model.pvalues.iloc[1:]
        worst_pval = pvalues.max()
        if worst_pval > threshold_out:
            worst_feature = pvalues.idxmax()
            included.remove(worst_feature)
            changed=True
            if verbose:
                print('Drop {:30} with p-value {:.6}'.format(worst_feature, worst_pval))
        if not changed:
            break
    return included

# 直接使用所有特徵進行逐步回歸
stepwise_features = stepwise_selection(X_scaled, y, verbose=True)
print('Stepwise 選出的特徵:', stepwise_features)

# 5. SHAP 解釋
if len(stepwise_features) == 0:
    print("警告：特徵選擇未選出任何特徵，使用所有特徵進行分析")
    stepwise_features = X.columns.tolist()

model = LinearRegression()
model.fit(X_scaled[stepwise_features], y)

explainer = shap.Explainer(model, X_scaled[stepwise_features])
shap_values = explainer(X_scaled[stepwise_features])

shap.summary_plot(shap_values, X_scaled[stepwise_features], show=False)
plt.title('SHAP Summary for Stepwise-selected Features')
plt.show()

# 6. 分組訓練與測試
teams = df['Team'].tolist()
np.random.seed(42)
test_teams = np.random.choice(teams, size=5, replace=False)
train_teams = [t for t in teams if t not in test_teams]

X_train = X_scaled[df['Team'].isin(train_teams)][stepwise_features]
y_train = y[df['Team'].isin(train_teams)]
X_test = X_scaled[df['Team'].isin(test_teams)][stepwise_features]
y_test = y[df['Team'].isin(test_teams)]

# 7. XGBoost 訓練與預測
xgb_model = xgb.XGBRegressor(n_estimators=100, random_state=42)
xgb_model.fit(X_train, y_train)

y_train_pred = xgb_model.predict(X_train)
y_test_pred = xgb_model.predict(X_test)

train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

print(f'訓練組 R2: {train_r2:.3f}')
print(f'測試組 R2: {test_r2:.3f}')

# 顯示測試組預測結果
result = pd.DataFrame({
    'Team': np.array(test_teams),
    '實際勝率': y_test.values,
    '預測勝率': y_test_pred,
    '誤差': y_test_pred - y_test.values
})
print(result)

# 8. XGBoost SHAP
explainer_xgb = shap.Explainer(xgb_model, X_train)
shap_values_xgb = explainer_xgb(X_test)
shap.summary_plot(shap_values_xgb, X_test, show=False)
plt.title('XGBoost SHAP Summary (Test Set)')
plt.show()


Forward Selection（前進選擇法）

In [ ]:
# 2. 載入資料
import pandas as pd
import numpy as np

import sys
from pathlib import Path
_root = Path.cwd().resolve()
for _ in range(10):
    if (_root / "src" / "data_layout.py").exists():
        break
    _root = _root.parent
sys.path.insert(0, str(_root / "src"))
from data_layout import resolve_processed_data_dir
_csv_dir = resolve_processed_data_dir(_root)
df = pd.read_csv(_csv_dir / "2025_metrics.csv")
if "Season" in df.columns:
    df = df[df["Season"] == 2025].reset_index(drop=True)

# 3. 準備特徵與目標
target = 'Win Rate'
exclude_cols = [c for c in ['Team', 'Season', target] if c in df.columns]
X = df.drop(columns=exclude_cols)
y = df[target]

# 4. 標準化資料
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

# 5. Forward Selection（往前回歸）
import statsmodels.api as sm

def forward_selection(X, y, threshold_in=0.05, verbose=True):
    included = []
    while True:
        changed = False
        excluded = list(set(X.columns) - set(included))
        new_pval = pd.Series(index=excluded, dtype=float)
        for new_col in excluded:
            model = sm.OLS(y, sm.add_constant(pd.DataFrame(X[included + [new_col]]))).fit()
            new_pval[new_col] = model.pvalues[new_col]
        if new_pval.empty:
            break
        best_pval = new_pval.min()
        if best_pval < threshold_in:
            best_feature = new_pval.idxmin()
            included.append(best_feature)
            changed = True
            if verbose:
                print('Add  {:30} with p-value {:.6}'.format(best_feature, best_pval))
        if not changed:
            break
    return included

forward_features = forward_selection(X_scaled, y, verbose=True)
print('Forward Selection 選出的特徵:', forward_features)

# 6. SHAP 解釋
import shap
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(X_scaled[forward_features], y)

explainer = shap.Explainer(model, X_scaled[forward_features])
shap_values = explainer(X_scaled[forward_features])

shap.summary_plot(shap_values, X_scaled[forward_features], show=False)
plt.title('SHAP Summary for Forward-selected Features')
plt.show()

# 7. 分組訓練與測試
from sklearn.model_selection import train_test_split

teams = df['Team'].tolist()
np.random.seed(42)
test_teams = np.random.choice(teams, size=5, replace=False)
train_teams = [t for t in teams if t not in test_teams]

X_train = X_scaled[df['Team'].isin(train_teams)][forward_features]
y_train = y[df['Team'].isin(train_teams)]
X_test = X_scaled[df['Team'].isin(test_teams)][forward_features]
y_test = y[df['Team'].isin(test_teams)]

# 8. XGBoost 訓練與預測
import xgboost as xgb
from sklearn.metrics import r2_score

xgb_model = xgb.XGBRegressor(n_estimators=100, random_state=42)
xgb_model.fit(X_train, y_train)

y_train_pred = xgb_model.predict(X_train)
y_test_pred = xgb_model.predict(X_test)

train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

print(f'訓練組 R2: {train_r2:.3f}')
print(f'測試組 R2: {test_r2:.3f}')

# 顯示測試組預測結果
result = pd.DataFrame({
    'Team': np.array(test_teams),
    '實際勝率': y_test.values,
    '預測勝率': y_test_pred,
    '誤差': y_test_pred - y_test.values
})
print(result)

# 9. XGBoost SHAP
explainer_xgb = shap.Explainer(xgb_model, X_train)
shap_values_xgb = explainer_xgb(X_test)
shap.summary_plot(shap_values_xgb, X_test, show=False)
plt.title('XGBoost SHAP Summary (Test Set)')
plt.show()


Backward Elimination（反向淘汰法）

In [ ]:
# 1. 載入資料
import pandas as pd
import numpy as np

import sys
from pathlib import Path
_root = Path.cwd().resolve()
for _ in range(10):
    if (_root / "src" / "data_layout.py").exists():
        break
    _root = _root.parent
sys.path.insert(0, str(_root / "src"))
from data_layout import resolve_processed_data_dir
_csv_dir = resolve_processed_data_dir(_root)
df = pd.read_csv(_csv_dir / "2025_metrics.csv")
if "Season" in df.columns:
    df = df[df["Season"] == 2025].reset_index(drop=True)

# 2. 準備特徵與目標
target = 'Win Rate'
exclude_cols = [c for c in ['Team', 'Season', target] if c in df.columns]
X = df.drop(columns=exclude_cols)
y = df[target]

# 3. 標準化資料
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

# 4. Backward Elimination（純往後回歸）
import statsmodels.api as sm

def backward_elimination(X, y, threshold_out=0.10, verbose=True):
    included = list(X.columns)
    while True:
        changed = False
        model = sm.OLS(y, sm.add_constant(pd.DataFrame(X[included]))).fit()
        pvalues = model.pvalues.iloc[1:]  # exclude intercept
        worst_pval = pvalues.max()
        if worst_pval > threshold_out:
            worst_feature = pvalues.idxmax()
            included.remove(worst_feature)
            changed = True
            if verbose:
                print('Drop {:30} with p-value {:.6}'.format(worst_feature, worst_pval))
        if not changed:
            break
    return included

stepwise_features = backward_elimination(X_scaled, y, verbose=True)
print('Backward Elimination 選出的特徵:', stepwise_features)

# 5. SHAP 解釋
import shap
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(X_scaled[stepwise_features], y)

explainer = shap.Explainer(model, X_scaled[stepwise_features])
shap_values = explainer(X_scaled[stepwise_features])

shap.summary_plot(shap_values, X_scaled[stepwise_features], show=False)
plt.title('SHAP Summary for Backward Elimination-selected Features')
plt.show()

# 6. 分組訓練與測試
teams = df['Team'].tolist()
np.random.seed(42)
test_teams = np.random.choice(teams, size=5, replace=False)
train_teams = [t for t in teams if t not in test_teams]

X_train = X_scaled[df['Team'].isin(train_teams)][stepwise_features]
y_train = y[df['Team'].isin(train_teams)]
X_test = X_scaled[df['Team'].isin(test_teams)][stepwise_features]
y_test = y[df['Team'].isin(test_teams)]

# 7. XGBoost 訓練與預測
import xgboost as xgb
from sklearn.metrics import r2_score

xgb_model = xgb.XGBRegressor(n_estimators=100, random_state=42)
xgb_model.fit(X_train, y_train)

y_train_pred = xgb_model.predict(X_train)
y_test_pred = xgb_model.predict(X_test)

train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

print(f'訓練組 R2: {train_r2:.3f}')
print(f'測試組 R2: {test_r2:.3f}')

# 顯示測試組預測結果
result = pd.DataFrame({
    'Team': np.array(test_teams),
    '實際勝率': y_test.values,
    '預測勝率': y_test_pred,
    '誤差': y_test_pred - y_test.values
})
print(result)

# 8. XGBoost SHAP
explainer_xgb = shap.Explainer(xgb_model, X_train)
shap_values_xgb = explainer_xgb(X_test)
shap.summary_plot(shap_values_xgb, X_test, show=False)
plt.title('XGBoost SHAP Summary (Test Set)')
plt.show()
